# Unified Revision Experiment Suite on Kaggle GPU (`R1C4` + `R2C5/R2C14` + `R2C7`)

Runs all remaining local Ollama experiments in **one single Kaggle `GPU T4 x2` session** (~20–30 min per model):
1. **`R1C4` Live End-to-End Pipeline**: 50 Python tasks × 3 policies (`1` Neutral+Plaintext, `3` Neutral+XML, `7` Polite+Plaintext) with **strict parser → `ast.parse` → isolated tests → up to 2 REAL error-feedback retries**.
   - *(Optional: attach your earlier `*.code.repeated_k3.jsonl` dataset and set `REUSE_K3_DIR` to reuse Attempt 0; otherwise it runs Attempt 0 fresh automatically.)*
2. **`R2C5` + `R2C14` Neutral Paraphrase & Length-Matched Neutral Controls**: 30 Python tasks × 2 new neutral variants (`neutral_paraphrase_short`, `neutral_length_matched`) = 60 calls.
3. **`R2C7` Non-Math SE Benchmark (External Validity)**: 15 verified non-math Python SE tasks (embedded below) × 3 policies = 45 calls.

*(Note: `R2C10` needs 0 new local calls because Local models already ran TOANMATH!)*

**Kaggle Setup:** Accelerator = **`GPU T4 x2`**, Internet = **ON**, Input Dataset = `jakenty1/vnmathproblems2026`.
**Download destination:** Put all 3 `/kaggle/working/*.jsonl` files into `14ImprovePhrase/results_revision/local/`.


In [ ]:
# === CONFIGURATION ===
MODEL_CHOICE = "qwen2.5:7b"  # Recommended: run "qwen2.5:7b" and "llama3.1:8b"
RUN_SUITES = ["pipeline", "paraphrase", "se_bench"]

POLICY_IDS = ["1", "3", "7"]  # 1=neutral_plaintext, 3=neutral_xml, 7=polite_plaintext
MAX_RETRIES = 2
PARAPHRASE_TASKS = 30
TEMPERATURE = 0.2
NUM_PREDICT = 2048
INTER_CALL_DELAY = 1.0
HTTP_TIMEOUT = 300
EVAL_TIMEOUT = 10
PYTHON_SYSTEM_PROMPT = "Output only the code, no explanations."

# Optional: path to folder containing {model}.code.repeated_k3.jsonl to reuse Attempt 0 for R1C4
REUSE_K3_DIR = None


In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import os, subprocess, time, requests
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], stdout=open("ollama_stdout.log", "w"), stderr=open("ollama_stderr.log", "w"))
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            print(f"Ollama ready after {i+1}s")
            break
    except Exception:
        pass
    time.sleep(1)
!ollama pull {MODEL_CHOICE}


In [ ]:
import ast, csv, json, re, sys, tempfile
from datetime import datetime, timezone
from pathlib import Path
from subprocess import TimeoutExpired

POLICY_LABELS = {"1": "neutral_plaintext", "3": "neutral_xml", "7": "polite_plaintext"}
FENCE_RE = re.compile(r"```(?:python|py)?\s*(.*?)```", re.S | re.I)
TEST_PREAMBLE = "from fractions import Fraction\nimport decimal\nimport itertools\nimport math\n"

NEUTRAL_PARAPHRASE_PREFIX = (
    "You are an AI assistant focused on delivering accurate, clear responses.\n\n"
    "Below is a Python programming specification. Implement the requested function "
    "accurately using standard production conventions, complete type annotations, "
    "and appropriate error handling.\n\nTask: "
)
NEUTRAL_LENGTH_MATCHED_PREFIX = (
    "You are an AI assistant dedicated to providing clear and accurate information to support any request.\n\n"
    "The specification below defines a standalone Python programming task that will be verified "
    "automatically against unit tests. Read the problem description, function signature, input "
    "constraints, and example assertions carefully before writing your implementation. Complete "
    "the required function precisely as specified using only the Python standard library. Follow "
    "standard production software engineering practices, including clean modular structure, full "
    "type annotations for parameters and return values, and proper handling of boundary cases.\n\nTask: "
)
SE_TEMPLATES = {
    "neutral_plaintext": "You are an AI assistant dedicated to providing clear and accurate information to support any request.\n\nTasks are provided below. Complete the following Python requirement precisely as specified. Follow standard production practices, including full type hints and error handling.\n\nTask: {problem}",
    "neutral_xml": "<prompt>\n<instruction>\nYou are an AI assistant dedicated to providing clear and accurate information to support any request.\n\nTasks are provided below. Complete the following Python requirement precisely as specified. Follow standard production practices, including full type hints and error handling.\n</instruction>\n<task>\n{problem}\n</task>\n</prompt>",
    "polite_plaintext": "You are an AI assistant dedicated to providing clear and accurate information to support any request.\n\nCould you please help me solve this Python problem? I would really appreciate it if you could provide a high-quality solution. I would be so grateful if you could ensure the code is modular, includes full type hints, and handles errors correctly. Thank you so much for your help!\n\nTask: {problem}",
}
SE_TASKS = json.loads("[\n  {\n    \"task_id\": \"se_01\",\n    \"domain\": \"parsing\",\n    \"prompt_body\": \"Write a Python function `parse_env_file(text: str) -> dict[str, str]` that parses `.env`-style configuration text. Ignore blank lines and lines starting with `#` (after stripping leading whitespace). Split each key-value line on the first `=`. Strip surrounding whitespace from keys and values, and if a value is enclosed in matching single or double quotes, remove the outer quotes. Return a dictionary mapping keys to values.\\n\\nExample assertions:\\nassert parse_env_file('HOST=localhost\\\\nPORT=\\\"8080\\\"') == {'HOST': 'localhost', 'PORT': '8080'}\\nassert parse_env_file('# comment\\\\nKEY=a=b=c') == {'KEY': 'a=b=c'}\",\n    \"reference\": \"def parse_env_file(text: str) -> dict[str, str]:\\n    out: dict[str, str] = {}\\n    for raw in text.splitlines():\\n        line = raw.strip()\\n        if not line or line.startswith('#') or '=' not in line:\\n            continue\\n        k, v = line.split('=', 1)\\n        k, v = k.strip(), v.strip()\\n        if len(v) >= 2 and v[0] == v[-1] and v[0] in ('\\\"', \\\"'\\\"):\\n            v = v[1:-1]\\n        out[k] = v\\n    return out\\n\",\n    \"tests\": [\n      \"assert parse_env_file('HOST=localhost\\\\nPORT=\\\"8080\\\"') == {'HOST': 'localhost', 'PORT': '8080'}\",\n      \"assert parse_env_file('# comment\\\\nKEY=a=b=c') == {'KEY': 'a=b=c'}\",\n      \"assert parse_env_file('  \\\\n  # indented\\\\nNAME = \\\\'app\\\\'  ') == {'NAME': 'app'}\",\n      \"assert parse_env_file('EMPTY=\\\\nA=1\\\\nA=2') == {'EMPTY': '', 'A': '2'}\",\n      \"assert parse_env_file('') == {}\"\n    ]\n  },\n  {\n    \"task_id\": \"se_02\",\n    \"domain\": \"versioning\",\n    \"prompt_body\": \"Write a Python function `compare_semver(v1: str, v2: str) -> int` that compares two semantic version strings of the form `MAJOR.MINOR.PATCH` (with an optional leading `v`). Return `-1` if `v1 < v2`, `0` if `v1 == v2`, and `1` if `v1 > v2`. Missing minor or patch components default to `0`.\\n\\nExample assertions:\\nassert compare_semver('v1.2.10', '1.2.2') == 1\\nassert compare_semver('2.0', '2.0.0') == 0\",\n    \"reference\": \"def compare_semver(v1: str, v2: str) -> int:\\n    def norm(v: str) -> tuple[int, int, int]:\\n        parts = [int(p) for p in v.strip().lstrip('vV').split('.')]\\n        while len(parts) < 3:\\n            parts.append(0)\\n        return (parts[0], parts[1], parts[2])\\n    a, b = norm(v1), norm(v2)\\n    return -1 if a < b else (1 if a > b else 0)\\n\",\n    \"tests\": [\n      \"assert compare_semver('v1.2.10', '1.2.2') == 1\",\n      \"assert compare_semver('2.0', '2.0.0') == 0\",\n      \"assert compare_semver('0.9.9', '1.0.0') == -1\",\n      \"assert compare_semver('v10.0.1', 'v2.99.99') == 1\",\n      \"assert compare_semver('3', 'v3.0.0') == 0\"\n    ]\n  },\n  {\n    \"task_id\": \"se_03\",\n    \"domain\": \"intervals\",\n    \"prompt_body\": \"Write a Python function `merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]` that merges all overlapping or touching integer intervals `(start, end)` (where `start <= end`) and returns them sorted by start.\\n\\nExample assertions:\\nassert merge_intervals([(1, 3), (2, 6), (8, 10)]) == [(1, 6), (8, 10)]\\nassert merge_intervals([(5, 7), (1, 5)]) == [(1, 7)]\",\n    \"reference\": \"def merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]:\\n    if not intervals:\\n        return []\\n    s = sorted(intervals)\\n    merged = [s[0]]\\n    for a, b in s[1:]:\\n        pa, pb = merged[-1]\\n        if a <= pb:\\n            merged[-1] = (pa, max(pb, b))\\n        else:\\n            merged.append((a, b))\\n    return merged\\n\",\n    \"tests\": [\n      \"assert merge_intervals([(1, 3), (2, 6), (8, 10)]) == [(1, 6), (8, 10)]\",\n      \"assert merge_intervals([(5, 7), (1, 5)]) == [(1, 7)]\",\n      \"assert merge_intervals([]) == []\",\n      \"assert merge_intervals([(1, 10), (2, 3), (4, 5)]) == [(1, 10)]\",\n      \"assert merge_intervals([(10, 12), (1, 2), (3, 4)]) == [(1, 2), (3, 4), (10, 12)]\"\n    ]\n  },\n  {\n    \"task_id\": \"se_04\",\n    \"domain\": \"caching\",\n    \"prompt_body\": \"Write a Python function `simulate_lru(capacity: int, ops: list[tuple]) -> list[int]` that simulates an LRU cache of positive integer `capacity`. Each operation in `ops` is either `('put', key, value)` or `('get', key)`. For each `'get'`, append the cached value (or `-1` if missing) to the returned list and mark the key as recently used. On `'put'`, insert/update the key, mark it recently used, and evict the least recently used key if capacity is exceeded.\\n\\nExample assertions:\\nassert simulate_lru(2, [('put', 1, 10), ('put', 2, 20), ('get', 1), ('put', 3, 30), ('get', 2)]) == [10, -1]\\nassert simulate_lru(1, [('put', 1, 5), ('put', 2, 6), ('get', 1), ('get', 2)]) == [-1, 6]\",\n    \"reference\": \"from collections import OrderedDict\\ndef simulate_lru(capacity: int, ops: list[tuple]) -> list[int]:\\n    cache: OrderedDict[int, int] = OrderedDict()\\n    out: list[int] = []\\n    for op in ops:\\n        if op[0] == 'get':\\n            k = op[1]\\n            if k in cache:\\n                cache.move_to_end(k)\\n                out.append(cache[k])\\n            else:\\n                out.append(-1)\\n        else:\\n            _, k, v = op\\n            cache[k] = v\\n            cache.move_to_end(k)\\n            if len(cache) > capacity:\\n                cache.popitem(last=False)\\n    return out\\n\",\n    \"tests\": [\n      \"assert simulate_lru(2, [('put', 1, 10), ('put', 2, 20), ('get', 1), ('put', 3, 30), ('get', 2)]) == [10, -1]\",\n      \"assert simulate_lru(1, [('put', 1, 5), ('put', 2, 6), ('get', 1), ('get', 2)]) == [-1, 6]\",\n      \"assert simulate_lru(2, [('put', 1, 1), ('put', 2, 2), ('put', 1, 10), ('put', 3, 3), ('get', 1), ('get', 2)]) == [10, -1]\",\n      \"assert simulate_lru(2, [('get', 99)]) == [-1]\",\n      \"assert simulate_lru(3, [('put', 1, 1), ('put', 2, 2), ('put', 3, 3), ('get', 1), ('put', 4, 4), ('get', 2), ('get', 3)]) == [1, -1, 3]\"\n    ]\n  },\n  {\n    \"task_id\": \"se_05\",\n    \"domain\": \"validation\",\n    \"prompt_body\": \"Write a Python function `is_balanced_brackets(s: str) -> bool` that checks whether all brackets `()`, `[]`, and `{}` in string `s` are properly balanced and nested. Ignore all non-bracket characters.\\n\\nExample assertions:\\nassert is_balanced_brackets('fn(a[0], {x: 1})') is True\\nassert is_balanced_brackets('([)]') is False\",\n    \"reference\": \"def is_balanced_brackets(s: str) -> bool:\\n    pairs = {')': '(', ']': '[', '}': '{'}\\n    opens = set(pairs.values())\\n    stack: list[str] = []\\n    for ch in s:\\n        if ch in opens:\\n            stack.append(ch)\\n        elif ch in pairs:\\n            if not stack or stack.pop() != pairs[ch]:\\n                return False\\n    return not stack\\n\",\n    \"tests\": [\n      \"assert is_balanced_brackets('fn(a[0], {x: 1})') is True\",\n      \"assert is_balanced_brackets('([)]') is False\",\n      \"assert is_balanced_brackets('') is True\",\n      \"assert is_balanced_brackets('(((') is False\",\n      \"assert is_balanced_brackets('a + b)(') is False\"\n    ]\n  },\n  {\n    \"task_id\": \"se_06\",\n    \"domain\": \"serialization\",\n    \"prompt_body\": \"Write a Python function `flatten_json(obj: dict, sep: str = '.') -> dict` that recursively flattens nested dictionaries into a single-level dictionary whose keys are joined by `sep`. Non-dict values (including lists, ints, strings, None, empty dicts `{}`) are kept as leaf values.\\n\\nExample assertions:\\nassert flatten_json({'a': {'b': 1, 'c': {'d': 2}}}) == {'a.b': 1, 'a.c.d': 2}\\nassert flatten_json({'x': [1, 2], 'y': {}}) == {'x': [1, 2], 'y': {}}\",\n    \"reference\": \"def flatten_json(obj: dict, sep: str = '.') -> dict:\\n    out = {}\\n    def walk(d: dict, prefix: str) -> None:\\n        for k, v in d.items():\\n            nk = f'{prefix}{sep}{k}' if prefix else str(k)\\n            if isinstance(v, dict) and v:\\n                walk(v, nk)\\n            else:\\n                out[nk] = v\\n    walk(obj, '')\\n    return out\\n\",\n    \"tests\": [\n      \"assert flatten_json({'a': {'b': 1, 'c': {'d': 2}}}) == {'a.b': 1, 'a.c.d': 2}\",\n      \"assert flatten_json({'x': [1, 2], 'y': {}}) == {'x': [1, 2], 'y': {}}\",\n      \"assert flatten_json({'a': {'b': None}}, sep='/') == {'a/b': None}\",\n      \"assert flatten_json({}) == {}\",\n      \"assert flatten_json({'a': 1, 'b': {'c': 2}}) == {'a': 1, 'b.c': 2}\"\n    ]\n  },\n  {\n    \"task_id\": \"se_07\",\n    \"domain\": \"logging\",\n    \"prompt_body\": \"Write a Python function `summarize_logs(lines: list[str]) -> dict[str, int]` that parses log lines formatted as `[LEVEL] message` (where LEVEL is uppercase letters like `INFO`, `WARN`, `ERROR`, with optional leading whitespace). Count occurrences of each level and return a dictionary `{level: count}`. Ignore malformed lines that do not start with `[LEVEL]`.\\n\\nExample assertions:\\nassert summarize_logs(['[INFO] started', '[ERROR] fail', '[INFO] done']) == {'INFO': 2, 'ERROR': 1}\\nassert summarize_logs(['bad line', '  [WARN] slow']) == {'WARN': 1}\",\n    \"reference\": \"import re\\ndef summarize_logs(lines: list[str]) -> dict[str, int]:\\n    counts: dict[str, int] = {}\\n    pat = re.compile(r'^\\\\s*\\\\[([A-Z]+)\\\\]\\\\s+')\\n    for ln in lines:\\n        m = pat.match(ln)\\n        if m:\\n            lvl = m.group(1)\\n            counts[lvl] = counts.get(lvl, 0) + 1\\n    return counts\\n\",\n    \"tests\": [\n      \"assert summarize_logs(['[INFO] started', '[ERROR] fail', '[INFO] done']) == {'INFO': 2, 'ERROR': 1}\",\n      \"assert summarize_logs(['bad line', '  [WARN] slow']) == {'WARN': 1}\",\n      \"assert summarize_logs([]) == {}\",\n      \"assert summarize_logs(['[debug] lower', '[DEBUG] ok']) == {'DEBUG': 1}\",\n      \"assert summarize_logs(['[ERROR] a', '[ERROR] b', '[WARN] c']) == {'ERROR': 2, 'WARN': 1}\"\n    ]\n  },\n  {\n    \"task_id\": \"se_08\",\n    \"domain\": \"compression\",\n    \"prompt_body\": \"Write a Python function `rle_encode(s: str) -> str` that performs run-length encoding on string `s`, replacing each maximal run of identical characters with `<char><count>` (even when count is 1). Return `''` for an empty string.\\n\\nExample assertions:\\nassert rle_encode('aaabbc') == 'a3b2c1'\\nassert rle_encode('') == ''\",\n    \"reference\": \"def rle_encode(s: str) -> str:\\n    if not s:\\n        return ''\\n    out = []\\n    cur, cnt = s[0], 1\\n    for ch in s[1:]:\\n        if ch == cur:\\n            cnt += 1\\n        else:\\n            out.append(f'{cur}{cnt}')\\n            cur, cnt = ch, 1\\n    out.append(f'{cur}{cnt}')\\n    return ''.join(out)\\n\",\n    \"tests\": [\n      \"assert rle_encode('aaabbc') == 'a3b2c1'\",\n      \"assert rle_encode('') == ''\",\n      \"assert rle_encode('x') == 'x1'\",\n      \"assert rle_encode('aaaa') == 'a4'\",\n      \"assert rle_encode('aba') == 'a1b1a1'\"\n    ]\n  },\n  {\n    \"task_id\": \"se_09\",\n    \"domain\": \"paths\",\n    \"prompt_body\": \"Write a Python function `normalize_unix_path(path: str) -> str` that simplifies an absolute Unix file path (handling repeated slashes `//`, `.`, and `..` segments). Always return a canonical path starting with `/` with no trailing slash (except root `'/'`).\\n\\nExample assertions:\\nassert normalize_unix_path('/a/./b/../../c/') == '/c'\\nassert normalize_unix_path('/../') == '/'\",\n    \"reference\": \"def normalize_unix_path(path: str) -> str:\\n    stack: list[str] = []\\n    for part in path.split('/'):\\n        if not part or part == '.':\\n            continue\\n        if part == '..':\\n            if stack:\\n                stack.pop()\\n        else:\\n            stack.append(part)\\n    return '/' + '/'.join(stack)\\n\",\n    \"tests\": [\n      \"assert normalize_unix_path('/a/./b/../../c/') == '/c'\",\n      \"assert normalize_unix_path('/../') == '/'\",\n      \"assert normalize_unix_path('/home//foo/') == '/home/foo'\",\n      \"assert normalize_unix_path('/a/b/c/..') == '/a/b'\",\n      \"assert normalize_unix_path('/') == '/'\"\n    ]\n  },\n  {\n    \"task_id\": \"se_10\",\n    \"domain\": \"graph_deps\",\n    \"prompt_body\": \"Write a Python function `topo_sort_tasks(deps: dict[str, list[str]]) -> list[str]` that returns a valid topological execution order for tasks where `deps[task]` lists the prerequisites that must run BEFORE `task`. Whenever multiple tasks are ready simultaneously, pick the lexicographically smallest task name first. If a cycle exists, return an empty list `[]`.\\n\\nExample assertions:\\nassert topo_sort_tasks({'b': ['a'], 'a': [], 'c': ['a']}) == ['a', 'b', 'c']\\nassert topo_sort_tasks({'a': ['b'], 'b': ['a']}) == []\",\n    \"reference\": \"import heapq\\ndef topo_sort_tasks(deps: dict[str, list[str]]) -> list[str]:\\n    nodes = set(deps.keys())\\n    for pres in deps.values():\\n        nodes.update(pres)\\n    indeg = {n: 0 for n in nodes}\\n    adj: dict[str, list[str]] = {n: [] for n in nodes}\\n    for task, pres in deps.items():\\n        for p in pres:\\n            adj[p].append(task)\\n            indeg[task] += 1\\n    heap = [n for n, d in indeg.items() if d == 0]\\n    heapq.heapify(heap)\\n    order: list[str] = []\\n    while heap:\\n        u = heapq.heappop(heap)\\n        order.append(u)\\n        for v in adj[u]:\\n            indeg[v] -= 1\\n            if indeg[v] == 0:\\n                heapq.heappush(heap, v)\\n    return order if len(order) == len(nodes) else []\\n\",\n    \"tests\": [\n      \"assert topo_sort_tasks({'b': ['a'], 'a': [], 'c': ['a']}) == ['a', 'b', 'c']\",\n      \"assert topo_sort_tasks({'a': ['b'], 'b': ['a']}) == []\",\n      \"assert topo_sort_tasks({'deploy': ['build', 'test'], 'test': ['build'], 'build': []}) == ['build', 'test', 'deploy']\",\n      \"assert topo_sort_tasks({'z': [], 'a': [], 'm': []}) == ['a', 'm', 'z']\",\n      \"assert topo_sort_tasks({}) == []\"\n    ]\n  },\n  {\n    \"task_id\": \"se_11\",\n    \"domain\": \"rate_limit\",\n    \"prompt_body\": \"Write a Python function `allow_requests(timestamps: list[int], window: int, limit: int) -> list[bool]` for a sliding-window rate limiter. Given non-decreasing integer `timestamps`, allow a request at time `t` if strictly fewer than `limit` previously ALLOWED requests occurred in the half-open window `(t - window, t]`. Return a list of booleans indicating whether each request is allowed.\\n\\nExample assertions:\\nassert allow_requests([1, 2, 3, 4], window=5, limit=2) == [True, True, False, False]\\nassert allow_requests([1, 2, 6, 7], window=5, limit=2) == [True, True, True, True]\",\n    \"reference\": \"from collections import deque\\ndef allow_requests(timestamps: list[int], window: int, limit: int) -> list[bool]:\\n    q: deque[int] = deque()\\n    res: list[bool] = []\\n    for t in timestamps:\\n        while q and q[0] <= t - window:\\n            q.popleft()\\n        if len(q) < limit:\\n            q.append(t)\\n            res.append(True)\\n        else:\\n            res.append(False)\\n    return res\\n\",\n    \"tests\": [\n      \"assert allow_requests([1, 2, 3, 4], window=5, limit=2) == [True, True, False, False]\",\n      \"assert allow_requests([1, 2, 6, 7], window=5, limit=2) == [True, True, True, True]\",\n      \"assert allow_requests([1, 1, 1], window=10, limit=2) == [True, True, False]\",\n      \"assert allow_requests([], window=5, limit=2) == []\",\n      \"assert allow_requests([0, 5, 10], window=5, limit=1) == [True, True, True]\"\n    ]\n  },\n  {\n    \"task_id\": \"se_12\",\n    \"domain\": \"urls\",\n    \"prompt_body\": \"Write a Python function `parse_query_params(url: str) -> dict[str, list[str]]` that extracts query parameters from a URL string (after `?` and before any `#` fragment), URL-decoding keys and values (treating `+` as space) and preserving the order of values for repeated keys. Return `{}` if there is no query string.\\n\\nExample assertions:\\nassert parse_query_params('https://ex.com/s?a=1&b=hello+world&a=2#top') == {'a': ['1', '2'], 'b': ['hello world']}\\nassert parse_query_params('https://ex.com/home') == {}\",\n    \"reference\": \"from urllib.parse import parse_qs, urlsplit\\ndef parse_query_params(url: str) -> dict[str, list[str]]:\\n    q = urlsplit(url).query\\n    return parse_qs(q, keep_blank_values=True) if q else {}\\n\",\n    \"tests\": [\n      \"assert parse_query_params('https://ex.com/s?a=1&b=hello+world&a=2#top') == {'a': ['1', '2'], 'b': ['hello world']}\",\n      \"assert parse_query_params('https://ex.com/home') == {}\",\n      \"assert parse_query_params('/path?x=1%202&y=') == {'x': ['1 2'], 'y': ['']}\",\n      \"assert parse_query_params('https://ex.com/#frag') == {}\",\n      \"assert parse_query_params('?k=v') == {'k': ['v']}\"\n    ]\n  },\n  {\n    \"task_id\": \"se_13\",\n    \"domain\": \"cli_parsing\",\n    \"prompt_body\": \"Write a Python function `parse_cli_flags(tokens: list[str]) -> dict[str, str | bool]` that parses CLI argument tokens. Flags start with `--`. If `--flag=value` is given, store `flag: value`. If `--flag` is followed by a non-flag token (not starting with `--`), store `flag: next_token`. Otherwise store `flag: True`. Ignore standalone positional tokens.\\n\\nExample assertions:\\nassert parse_cli_flags(['--host', '0.0.0.0', '--verbose', '--port=8080']) == {'host': '0.0.0.0', 'verbose': True, 'port': '8080'}\\nassert parse_cli_flags(['--dry-run']) == {'dry-run': True}\",\n    \"reference\": \"def parse_cli_flags(tokens: list[str]) -> dict[str, str | bool]:\\n    out: dict[str, str | bool] = {}\\n    i = 0\\n    while i < len(tokens):\\n        tok = tokens[i]\\n        if tok.startswith('--') and len(tok) > 2:\\n            body = tok[2:]\\n            if '=' in body:\\n                k, v = body.split('=', 1)\\n                out[k] = v\\n                i += 1\\n            elif i + 1 < len(tokens) and not tokens[i + 1].startswith('--'):\\n                out[body] = tokens[i + 1]\\n                i += 2\\n            else:\\n                out[body] = True\\n                i += 1\\n        else:\\n            i += 1\\n    return out\\n\",\n    \"tests\": [\n      \"assert parse_cli_flags(['--host', '0.0.0.0', '--verbose', '--port=8080']) == {'host': '0.0.0.0', 'verbose': True, 'port': '8080'}\",\n      \"assert parse_cli_flags(['--dry-run']) == {'dry-run': True}\",\n      \"assert parse_cli_flags(['pos', '--a', '--b', 'val']) == {'a': True, 'b': 'val'}\",\n      \"assert parse_cli_flags(['--k=1=2']) == {'k': '1=2'}\",\n      \"assert parse_cli_flags([]) == {}\"\n    ]\n  },\n  {\n    \"task_id\": \"se_14\",\n    \"domain\": \"text_processing\",\n    \"prompt_body\": \"Write a Python function `extract_markdown_links(md: str) -> list[tuple[str, str]]` that extracts all inline Markdown links `[label](url)` (excluding images prefixed with `![label](url)`) and returns them in order as `(label, url)` tuples.\\n\\nExample assertions:\\nassert extract_markdown_links('See [Docs](https://a.com) and ![img](pic.png)') == [('Docs', 'https://a.com')]\\nassert extract_markdown_links('[A](1) [B](2)') == [('A', '1'), ('B', '2')]\",\n    \"reference\": \"import re\\ndef extract_markdown_links(md: str) -> list[tuple[str, str]]:\\n    return re.findall(r'(?<!!)\\\\[([^\\\\]]+)\\\\]\\\\(([^)]+)\\\\)', md)\\n\",\n    \"tests\": [\n      \"assert extract_markdown_links('See [Docs](https://a.com) and ![img](pic.png)') == [('Docs', 'https://a.com')]\",\n      \"assert extract_markdown_links('[A](1) [B](2)') == [('A', '1'), ('B', '2')]\",\n      \"assert extract_markdown_links('plain text') == []\",\n      \"assert extract_markdown_links('![only](img.jpg)') == []\",\n      \"assert extract_markdown_links('Start [one](u1), then ![pic](p), end [two](u2).') == [('one', 'u1'), ('two', 'u2')]\"\n    ]\n  },\n  {\n    \"task_id\": \"se_15\",\n    \"domain\": \"data_cleaning\",\n    \"prompt_body\": \"Write a Python function `dedupe_preserve_order(items: list[dict], key: str) -> list[dict]` that deduplicates a list of dictionaries by the value of `item[key]`, keeping the FIRST occurrence of each distinct key value and preserving original order. Skip any dictionary missing `key`.\\n\\nExample assertions:\\nassert dedupe_preserve_order([{'id': 1, 'v': 'a'}, {'id': 2, 'v': 'b'}, {'id': 1, 'v': 'c'}], 'id') == [{'id': 1, 'v': 'a'}, {'id': 2, 'v': 'b'}]\\nassert dedupe_preserve_order([{'v': 1}, {'id': 1}], 'id') == [{'id': 1}]\",\n    \"reference\": \"def dedupe_preserve_order(items: list[dict], key: str) -> list[dict]:\\n    seen = set()\\n    out = []\\n    for item in items:\\n        if key not in item:\\n            continue\\n        val = item[key]\\n        if val not in seen:\\n            seen.add(val)\\n            out.append(item)\\n    return out\\n\",\n    \"tests\": [\n      \"assert dedupe_preserve_order([{'id': 1, 'v': 'a'}, {'id': 2, 'v': 'b'}, {'id': 1, 'v': 'c'}], 'id') == [{'id': 1, 'v': 'a'}, {'id': 2, 'v': 'b'}]\",\n      \"assert dedupe_preserve_order([{'v': 1}, {'id': 1}], 'id') == [{'id': 1}]\",\n      \"assert dedupe_preserve_order([], 'id') == []\",\n      \"assert dedupe_preserve_order([{'k': 'x'}, {'k': 'y'}, {'k': 'x'}], 'k') == [{'k': 'x'}, {'k': 'y'}]\",\n      \"assert dedupe_preserve_order([{'id': 0}, {'id': 0}], 'id') == [{'id': 0}]\"\n    ]\n  }\n]\n")

def find_problems_csv():
    for p in ["/kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv", "/kaggle/input/vnmathproblems2026/problems.csv"]:
        if os.path.exists(p): return p
    for d, _, fns in os.walk("/kaggle/input"):
        if "problems.csv" in fns: return os.path.join(d, "problems.csv")
    raise FileNotFoundError("problems.csv not found")

def strict_extract_code(text: str) -> str:
    if not text: return ""
    m = FENCE_RE.search(text)
    if m: return m.group(1).strip()
    s = text.strip()
    return s if s.startswith(("def ", "import ", "from ")) else ""

def run_downstream_consumer(response_text: str, tests: list[str]) -> dict:
    t0 = time.perf_counter()
    if not response_text or not response_text.strip():
        return {"passed": False, "stage": "empty_response", "error_summary": "Empty response", "parse_ms": round((time.perf_counter()-t0)*1000, 3), "validate_ms": 0.0, "exec_ms": 0.0}
    code = strict_extract_code(response_text)
    parse_ms = round((time.perf_counter()-t0)*1000, 3)
    if not code:
        return {"passed": False, "stage": "parse_fail", "error_summary": "No fenced Python block or bare definition", "parse_ms": parse_ms, "validate_ms": 0.0, "exec_ms": 0.0}
    tv = time.perf_counter()
    try: ast.parse(code)
    except SyntaxError as exc:
        return {"passed": False, "stage": "syntax_fail", "error_summary": f"SyntaxError: {exc.msg} (line {exc.lineno})", "parse_ms": parse_ms, "validate_ms": round((time.perf_counter()-tv)*1000, 3), "exec_ms": 0.0}
    validate_ms = round((time.perf_counter()-tv)*1000, 3)
    program = TEST_PREAMBLE + "\n" + code + "\n\n" + "\n".join(tests) + "\nprint('PASS')\n"
    te = time.perf_counter()
    with tempfile.TemporaryDirectory() as tmpdir:
        sp = Path(tmpdir) / "candidate.py"
        sp.write_text(program, encoding="utf-8")
        try: proc = subprocess.run([sys.executable, str(sp)], capture_output=True, text=True, timeout=EVAL_TIMEOUT)
        except TimeoutExpired:
            return {"passed": False, "stage": "timeout", "error_summary": f"Timed out after {EVAL_TIMEOUT}s", "parse_ms": parse_ms, "validate_ms": validate_ms, "exec_ms": round((time.perf_counter()-te)*1000, 3)}
    exec_ms = round((time.perf_counter()-te)*1000, 3)
    if proc.returncode == 0:
        return {"passed": True, "stage": "ok", "error_summary": "", "parse_ms": parse_ms, "validate_ms": validate_ms, "exec_ms": exec_ms}
    lines = (proc.stderr or proc.stdout or "Failed").strip().splitlines()
    stage = "assertion_fail" if "AssertionError" in (proc.stderr or "") else "runtime_fail"
    return {"passed": False, "stage": stage, "error_summary": (lines[-1] if lines else stage)[:300], "parse_ms": parse_ms, "validate_ms": validate_ms, "exec_ms": exec_ms}

def build_retry_prompt(orig_prompt: str, prev_stage: str, err_summary: str) -> str:
    if prev_stage in {"empty_response", "parse_fail", "syntax_fail"}:
        g = f"Your previous output failed downstream validation ({prev_stage}: {err_summary}). Output ONLY valid, runnable Python code inside a single ```python ... ``` block with no surrounding prose or explanations."
    else:
        g = f"Your previous Python solution failed automated testing ({prev_stage}: {err_summary}). Fix the bug and output ONLY the corrected, runnable Python code inside a single ```python ... ``` block with no explanations."
    return f"{orig_prompt}\n\n[Pipeline Recovery Feedback]\n{g}"

def extract_task_payload(prompt: str) -> str:
    i = prompt.find("Task:")
    return prompt[i+5:].strip() if i >= 0 else prompt.strip()

def call_ollama(model: str, prompt: str):
    t0 = time.perf_counter()
    payload = {"model": model, "prompt": prompt, "system": PYTHON_SYSTEM_PROMPT, "stream": False, "options": {"temperature": TEMPERATURE, "num_predict": NUM_PREDICT}}
    r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    d = r.json()
    if INTER_CALL_DELAY > 0: time.sleep(INTER_CALL_DELAY)
    return d.get("response", ""), round((time.perf_counter()-t0)*1000, 3), d.get("prompt_eval_count", 0), d.get("eval_count", 0)

def load_done(path: str) -> set[str]:
    done = set()
    if os.path.exists(path):
        for ln in open(path, encoding="utf-8"):
            if ln.strip():
                try: done.add(json.loads(ln)["item_key"])
                except Exception: pass
    return done


In [ ]:
model_slug = re.sub(r"[^A-Za-z0-9_.-]+", "-", MODEL_CHOICE).strip("-").lower()
all_rows = list(csv.DictReader(open(find_problems_csv(), encoding="utf-8")))

# --- SUITE 1: R1C4 Live End-to-End Pipeline with Real Retries ---
if "pipeline" in RUN_SUITES:
    out_p = f"/kaggle/working/{model_slug}.r1c4_pipeline.jsonl"
    done = load_done(out_p)
    reusable = {}
    if REUSE_K3_DIR:
        cand = os.path.join(REUSE_K3_DIR, f"{model_slug}.code.repeated_k3.jsonl")
        if os.path.exists(cand):
            for ln in open(cand, encoding="utf-8"):
                r = json.loads(ln)
                if r.get("error") is None and int(r.get("rep_id", 0)) == 1:
                    reusable[(str(r["task_id"]), str(r["category_id"]))] = r
    rows = [r for r in all_rows if str(r["category_id"]) in POLICY_IDS and f"{r['task_id']}::{r['category_id']}" not in done]
    print(f"\n=== [R1C4 pipeline][{MODEL_CHOICE}] remaining={len(rows)}, reusable_rep1={len(reusable)} ===")
    with open(out_p, "a", encoding="utf-8") as out:
        for idx, row in enumerate(rows, start=1):
            tid, cid = str(row["task_id"]), str(row["category_id"])
            ikey = f"{tid}::{cid}"
            tests = json.loads(row.get("test_cases", "[]")) + json.loads(row.get("challenge_test_list", "[]"))
            attempts, passed_at = [], None
            prev = reusable.get((tid, cid))
            if prev:
                resp0, llm0, ptok0, ctok0, reused = str(prev.get("response_text", "")), round(float(prev.get("duration_seconds", 0))*1000, 3), int(prev.get("prompt_tokens") or 0), int(prev.get("completion_tokens") or 0), True
            else:
                resp0, llm0, ptok0, ctok0 = call_ollama(MODEL_CHOICE, row["prompt"].strip())
                reused = False
            c0 = run_downstream_consumer(resp0, tests)
            attempts.append({"attempt": 0, "reused_from_k3": reused, **c0, "llm_ms": llm0, "prompt_tokens": ptok0, "completion_tokens": ctok0, "response_text": resp0})
            if c0["passed"]: passed_at = 0
            else:
                ps, pe = c0["stage"], c0["error_summary"]
                for r_idx in range(1, MAX_RETRIES + 1):
                    rr, lr, pr, cr_tok = call_ollama(MODEL_CHOICE, build_retry_prompt(row["prompt"].strip(), ps, pe))
                    cr = run_downstream_consumer(rr, tests)
                    attempts.append({"attempt": r_idx, "reused_from_k3": False, **cr, "llm_ms": lr, "prompt_tokens": pr, "completion_tokens": cr_tok, "response_text": rr})
                    if cr["passed"]: passed_at = r_idx; break
                    ps, pe = cr["stage"], cr["error_summary"]
            rec = {"timestamp": datetime.now(timezone.utc).isoformat(), "suite": "r1c4_pipeline", "regime": "local", "model": MODEL_CHOICE, "model_label": model_slug, "item_key": ikey, "task_id": tid, "category_id": cid, "policy_label": POLICY_LABELS[cid], "attempts_used": len(attempts), "retries_used": len(attempts)-1, "passed_r0": bool(attempts[0]["passed"]), "passed_r1": bool(passed_at is not None and passed_at <= 1), "passed_r2": bool(passed_at is not None and passed_at <= 2), "initial_stage": attempts[0]["stage"], "final_stage": attempts[-1]["stage"], "total_prompt_tokens": sum(a["prompt_tokens"] for a in attempts), "total_completion_tokens": sum(a["completion_tokens"] for a in attempts), "llm_total_ms": round(sum(a["llm_ms"] for a in attempts), 3), "consumer_total_ms": round(sum(a["parse_ms"]+a["validate_ms"]+a["exec_ms"] for a in attempts), 3), "api_error": None, "attempts": attempts}
            out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
            if idx % 10 == 0 or idx == len(rows): print(f"[R1C4][{idx}/{len(rows)}] {ikey}: r0={rec['passed_r0']} r2={rec['passed_r2']} att={rec['attempts_used']}", flush=True)

# --- SUITE 2: R2C5 + R2C14 Neutral Paraphrase & Length-Matched Neutral Controls ---
if "paraphrase" in RUN_SUITES:
    out_p = f"/kaggle/working/{model_slug}.r2c5_r2c14_paraphrase.jsonl"
    done = load_done(out_p)
    base_rows = [r for r in all_rows if str(r["category_id"]) == "1"][:PARAPHRASE_TASKS]
    sched = [(vn, pref, r) for vn, pref in [("neutral_paraphrase_short", NEUTRAL_PARAPHRASE_PREFIX), ("neutral_length_matched", NEUTRAL_LENGTH_MATCHED_PREFIX)] for r in base_rows if f"{r['task_id']}::{vn}" not in done]
    print(f"\n=== [R2C5/R2C14 paraphrase][{MODEL_CHOICE}] remaining={len(sched)} ===")
    with open(out_p, "a", encoding="utf-8") as out:
        for idx, (vn, pref, row) in enumerate(sched, start=1):
            tid = str(row["task_id"]); ikey = f"{tid}::{vn}"
            tests = json.loads(row.get("test_cases", "[]")) + json.loads(row.get("challenge_test_list", "[]"))
            resp, llm_ms, ptok, ctok = call_ollama(MODEL_CHOICE, pref + extract_task_payload(row["prompt"]))
            cons = run_downstream_consumer(resp, tests)
            rec = {"timestamp": datetime.now(timezone.utc).isoformat(), "suite": "r2c5_r2c14_paraphrase", "regime": "local", "model": MODEL_CHOICE, "model_label": model_slug, "item_key": ikey, "task_id": tid, "variant": vn, "passed": cons["passed"], "stage": cons["stage"], "llm_ms": llm_ms, "prompt_tokens": ptok, "completion_tokens": ctok, "api_error": None, "response_text": resp}
            out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
            if idx % 10 == 0 or idx == len(sched): print(f"[R2C5/14][{idx}/{len(sched)}] {ikey}: {'PASS' if cons['passed'] else 'FAIL'}", flush=True)

# --- SUITE 3: R2C7 Non-Math SE Benchmark (15 verified tasks x 3 policies = 45 calls) ---
if "se_bench" in RUN_SUITES:
    out_p = f"/kaggle/working/{model_slug}.r2c7_se_bench.jsonl"
    done = load_done(out_p)
    sched = [(t, pn, tmpl) for pn, tmpl in SE_TEMPLATES.items() for t in SE_TASKS if f"{t['task_id']}::{pn}" not in done]
    print(f"\n=== [R2C7 se_bench][{MODEL_CHOICE}] remaining={len(sched)} ===")
    with open(out_p, "a", encoding="utf-8") as out:
        for idx, (task, pn, tmpl) in enumerate(sched, start=1):
            tid = str(task["task_id"]); ikey = f"{tid}::{pn}"
            resp, llm_ms, ptok, ctok = call_ollama(MODEL_CHOICE, tmpl.format(problem=task["prompt_body"]))
            cons = run_downstream_consumer(resp, task["tests"])
            rec = {"timestamp": datetime.now(timezone.utc).isoformat(), "suite": "r2c7_se_bench", "regime": "local", "model": MODEL_CHOICE, "model_label": model_slug, "item_key": ikey, "task_id": tid, "domain": task["domain"], "policy_label": pn, "passed": cons["passed"], "stage": cons["stage"], "llm_ms": llm_ms, "prompt_tokens": ptok, "completion_tokens": ctok, "api_error": None, "response_text": resp}
            out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
            if idx % 10 == 0 or idx == len(sched): print(f"[R2C7][{idx}/{len(sched)}] {ikey}: {'PASS' if cons['passed'] else 'FAIL'}", flush=True)

print("\nALL SELECTED SUITES FINISHED! Download /kaggle/working/*.jsonl to 14ImprovePhrase/results_revision/local/")
